[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davis-mironga/kitui-washlab-analysis/blob/main/notebooks/02_Water_Stress_Index.ipynb)

# Notebook 02 — Water Access Stress Index (WASI)
**Project:** WASHLAB Climate-Smart WASH Pilot — Kitui County  
**Analyst:** Davis Mironga  
**Purpose:** Construct the 5-component Water Access Stress Index, aggregate to ward level, and export for mapping.

**Requires:** All raster exports from Notebook 01 completed in Google Drive.

---
## WASI Components
| # | Component | Weight | Raster file |
|---|-----------|--------|-------------|
| C1 | Distance to nearest functional GPS-verified borehole | 30% | Computed here from borehole master dataset |
| C2 | Rainfall deficit (baseline minus recent) | 25% | `kitui_rainfall_baseline_1981_2010.tif` vs `kitui_rainfall_recent_2020_2024.tif` |
| C3 | NDVI below baseline | 15% | `kitui_ndvi_mean_2000_2025.tif` vs `kitui_ndvi_baseline_2000_2004.tif` |
| C4 | Population density | 20% | `kitui_worldpop_2020.tif` |
| C5 | Terrain / slope | 10% | `kitui_slope_deg.tif` |

All components normalised 0→1 (0 = no stress, 1 = maximum stress) before weighting.

---
## Outputs
- `kitui_wasi_500m.tif` — composite raster at 500m resolution
- `kitui_wasi_c1_distance.tif` … `kitui_wasi_c5_slope.tif` — individual components
- `kitui_wasi_ward_table.csv` — mean WASI per ward with component breakdown
- `kitui_wasi_ward.geojson` — ward-level choropleth for Streamlit app

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
!pip install geopandas rasterio rasterstats scipy -q

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.transform import from_bounds
from rasterio.features import rasterize
from rasterio.enums import Resampling
from rasterio.warp import reproject
import rasterstats
from scipy.ndimage import distance_transform_edt
from shapely.geometry import Point, mapping
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from google.colab import drive

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/Kitui_WASHLAB/'
SAT   = DRIVE + 'satellite/'
OUT   = DRIVE + 'outputs/'
os.makedirs(OUT, exist_ok=True)

WGS84   = 'EPSG:4326'
UTM_CRS = 'EPSG:32637'  # UTM Zone 37N — Kenya, for accurate distance in metres

# Output grid: 500m pixels covering Kitui County (WGS84)
BOUNDS     = {'west': 37.6, 'east': 39.1, 'south': -2.2, 'north': -0.2}
PIXEL_DEG  = 500 / 111320   # ~0.00449° ≈ 500m at the equator
GRID_W     = int((BOUNDS['east']  - BOUNDS['west'])  / PIXEL_DEG)
GRID_H     = int((BOUNDS['north'] - BOUNDS['south']) / PIXEL_DEG)
GRID_TRANS = from_bounds(
    BOUNDS['west'], BOUNDS['south'], BOUNDS['east'], BOUNDS['north'],
    GRID_W, GRID_H
)
GRID_SHAPE = (GRID_H, GRID_W)

print('Setup complete')
print(f'Output grid: {GRID_W} x {GRID_H} pixels at ~500m')
print('Ensure all Notebook 01 exports are COMPLETED before continuing')

In [ ]:
# ── 1. Load borehole master dataset ───────────────────────────────────────────
df = pd.read_excel(
    DRIVE + 'Kitui_Boreholes_Master_Dataset.xlsx',
    sheet_name='B_Spatial_Analysis',
    header=2
)

# GPS-verified functional boreholes only — these define service coverage
# Non-functional and GPS-review records are excluded from C1 distance calculation
functional_verified = df[
    (df['Is_Functional'] == True) &
    (df['GPS_Quality']   == 'Verified')
].copy()

gdf_func = gpd.GeoDataFrame(
    functional_verified,
    geometry=[Point(xy) for xy in zip(
        functional_verified['Longitude'],
        functional_verified['Latitude']
    )],
    crs=WGS84
)

print(f'Total boreholes:                    {len(df)}')
print(f'Functional + GPS-verified (for C1): {len(gdf_func)}')
print(f'Wards with at least one BH:         {gdf_func["Ward"].nunique()} of 40')
print(f'Wards with no verified functional BH: {40 - gdf_func["Ward"].nunique()}')

In [ ]:
# ── C1: Distance to nearest functional borehole ────────────────────────────────
#
# Method:
#   1. Rasterise borehole point locations onto the 500m WGS84 grid
#   2. Run scipy Euclidean distance transform (in pixel units)
#   3. Convert pixel distance to kilometres using pixel size
#
# Note: PIXEL_DEG × 111.32 km/deg gives an equator-approximation.
# For Kitui (lat ≈ -1.3°) this is accurate to within ~0.1%.
# A more exact method would reproject to UTM, run the DT, then reproject back.

# Rasterise: pixels containing a borehole = 1, all others = 0
bh_shapes = [(mapping(geom), 1) for geom in gdf_func.geometry]
bh_grid   = rasterize(
    bh_shapes,
    out_shape=GRID_SHAPE,
    transform=GRID_TRANS,
    fill=0,
    dtype='uint8'
)

# Euclidean distance transform — output is distance in pixels
dist_px = distance_transform_edt(bh_grid == 0)

# Convert to km
pixel_km = PIXEL_DEG * 111.32
dist_km  = (dist_px * pixel_km).astype(np.float32)

print(f'C1 distance raster ready')
print(f'  Max distance:  {dist_km.max():.1f} km')
print(f'  Mean distance: {dist_km.mean():.1f} km')
print(f'  Within 2km:    {(dist_km < 2).mean() * 100:.1f}% of county pixels')
print(f'  Within 5km:    {(dist_km < 5).mean() * 100:.1f}% of county pixels')

In [ ]:
# ── Load satellite rasters ─────────────────────────────────────────────────────
#
# All rasters are reprojected and resampled to the common 500m WGS84 grid
# using bilinear interpolation.

def load_raster(path, target_transform=GRID_TRANS, target_shape=GRID_SHAPE):
    """
    Load a GeoTIFF and reproject/resample to the common output grid.
    Returns a 2D float32 array; NaN where nodata.
    """
    out = np.full(target_shape, np.nan, dtype=np.float32)
    with rasterio.open(path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=out,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=target_transform,
            dst_crs=WGS84,
            resampling=Resampling.bilinear,
            src_nodata=src.nodata,
            dst_nodata=np.nan
        )
    return out

print('Loading rasters from Notebook 01 exports...')

# C2 — Rainfall
rain_baseline = load_raster(SAT + 'kitui_rainfall_baseline_1981_2010.tif')
rain_recent   = load_raster(SAT + 'kitui_rainfall_recent_2020_2024.tif')

# C3 — NDVI (mean and baseline)
ndvi_current  = load_raster(SAT + 'kitui_ndvi_mean_2000_2025.tif')
ndvi_baseline = load_raster(SAT + 'kitui_ndvi_baseline_2000_2004.tif')

# C4 — Population
population    = load_raster(SAT + 'kitui_worldpop_2020.tif')

# C5 — Slope
slope         = load_raster(SAT + 'kitui_slope_deg.tif')

print('All rasters loaded. Value ranges:')
for name, arr in [
    ('Rainfall baseline (mm/yr)', rain_baseline),
    ('Rainfall recent  (mm/yr)', rain_recent),
    ('NDVI current',              ndvi_current),
    ('NDVI baseline',             ndvi_baseline),
    ('Population (100m)',         population),
    ('Slope (deg)',               slope),
]:
    v = arr[~np.isnan(arr)]
    print(f'  {name:30s}: min={v.min():.3f}  max={v.max():.3f}  mean={v.mean():.3f}')

In [ ]:
# ── Compute component stress layers ───────────────────────────────────────────

def normalise(arr, invert=False, pct_lo=2, pct_hi=98):
    """
    Min-max normalise to [0, 1] using percentile clipping to suppress outliers.
    invert=True flips the scale (high raw value → low stress).
    """
    a  = np.where(np.isnan(arr), np.nan, arr.astype(np.float32))
    lo = np.nanpercentile(a, pct_lo)
    hi = np.nanpercentile(a, pct_hi)
    a  = np.clip(a, lo, hi)
    n  = (a - lo) / (hi - lo + 1e-9)
    return (1.0 - n) if invert else n

# C1: Distance — far from a borehole = high stress
c1_dist = normalise(dist_km, invert=False)

# C2: Rainfall deficit — baseline minus recent (positive = drying trend = stress)
#     Only penalise pixels where recent < baseline (clip negatives to 0)
rain_deficit = np.where(
    (~np.isnan(rain_baseline)) & (~np.isnan(rain_recent)),
    np.clip(rain_baseline - rain_recent, 0, None),
    np.nan
)
c2_rain = normalise(rain_deficit, invert=False)

# C3: NDVI anomaly — current below baseline = vegetation stress
#     Only penalise pixels where current NDVI < baseline (clip positives to 0)
ndvi_deficit = np.where(
    (~np.isnan(ndvi_current)) & (~np.isnan(ndvi_baseline)),
    np.clip(ndvi_baseline - ndvi_current, 0, None),
    np.nan
)
c3_ndvi = normalise(ndvi_deficit, invert=False)

# C4: Population density — more people = higher exposure to stress
pop_clean = np.where(population <= 0, np.nan, population)  # strip nodata (<0)
c4_pop    = normalise(pop_clean, invert=False)

# C5: Slope — steeper = harder to reach water sources
c5_slope  = normalise(slope, invert=False)

print('Component stress layers computed:')
for name, c in [
    ('C1 Distance (30%)',          c1_dist),
    ('C2 Rainfall deficit (25%)',  c2_rain),
    ('C3 NDVI anomaly (15%)',      c3_ndvi),
    ('C4 Population (20%)',        c4_pop),
    ('C5 Slope (10%)',             c5_slope),
]:
    v = c[~np.isnan(c)]
    print(f'  {name:30s}: mean={v.mean():.3f}  std={v.std():.3f}')

# Diagnostic: how large is the NDVI anomaly?
valid_deficit = ndvi_deficit[~np.isnan(ndvi_deficit)]
print(f'\nNDVI anomaly (baseline - current):')
print(f'  {(valid_deficit > 0).mean()*100:.1f}% of pixels show vegetation below baseline')
print(f'  Mean deficit where positive: {valid_deficit[valid_deficit > 0].mean():.4f} NDVI units')

In [ ]:
# ── WASI composite ────────────────────────────────────────────────────────────

WEIGHTS = {'distance': 0.30, 'rainfall': 0.25, 'ndvi': 0.15,
           'population': 0.20, 'slope': 0.10}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9, 'Weights must sum to 1.0'

components = [c1_dist, c2_rain, c3_ndvi, c4_pop, c5_slope]
weights    = list(WEIGHTS.values())

# Weighted average — NaN pixels do not contribute weight
# (avoids pulling down WASI in edge pixels where one raster has no data)
total  = np.zeros(GRID_SHAPE, dtype=np.float32)
w_sum  = np.zeros(GRID_SHAPE, dtype=np.float32)
for arr, w in zip(components, weights):
    valid  = ~np.isnan(arr)
    total  = np.where(valid, total + arr * w, total)
    w_sum  = np.where(valid, w_sum + w, w_sum)

wasi = np.where(w_sum > 0.5,          # require at least 50% of weights to be valid
                total / w_sum,
                np.nan).astype(np.float32)

v = wasi[~np.isnan(wasi)]
print('WASI composite ready')
print(f'  Min:  {v.min():.3f}')
print(f'  Max:  {v.max():.3f}')
print(f'  Mean: {v.mean():.3f}')
print(f'  Std:  {v.std():.3f}')
print()
print('Stress class distribution:')
for label, lo, hi in [('Very High', 0.70, 1.01), ('High', 0.55, 0.70),
                       ('Moderate', 0.40, 0.55), ('Low', 0.25, 0.40),
                       ('Very Low', 0.00, 0.25)]:
    pct = ((v >= lo) & (v < hi)).mean() * 100
    print(f'  {label:10s} ({lo:.2f}–{hi:.2f}): {pct:.1f}% of valid pixels')

In [ ]:
# ── Export rasters ─────────────────────────────────────────────────────────────

def save_tif(path, array, transform=GRID_TRANS, crs=WGS84):
    with rasterio.open(
        path, 'w', driver='GTiff',
        height=GRID_H, width=GRID_W,
        count=1, dtype='float32',
        crs=crs, transform=transform,
        nodata=float('nan'), compress='lzw'
    ) as dst:
        dst.write(array.astype(np.float32), 1)

wasi_path = OUT + 'kitui_wasi_500m.tif'
save_tif(wasi_path, wasi)
print(f'WASI composite: {wasi_path}')

component_exports = {
    'c1_distance':   c1_dist,
    'c2_rainfall':   c2_rain,
    'c3_ndvi':       c3_ndvi,
    'c4_population': c4_pop,
    'c5_slope':      c5_slope,
}
for name, arr in component_exports.items():
    p = OUT + f'kitui_wasi_{name}.tif'
    save_tif(p, arr)
    print(f'  Component {name}: {p}')

In [ ]:
# ── Load ward boundaries ───────────────────────────────────────────────────────
#
# GADM Level 3 for Kitui (or county shapefile — see docs/data_dictionary.md).
# If ward names don't match the borehole dataset, run the crosswalk in Notebook 06
# and apply it here before the merge.

wards = gpd.read_file(DRIVE + 'boundaries/kitui_wards.shp').to_crs(WGS84)

# Standardise ward name column (GADM uses NAME_3 or ADM3_EN)
for col in ['NAME_3', 'ADM3_EN', 'Ward', 'ward_name', 'NAME']:
    if col in wards.columns:
        wards = wards.rename(columns={col: 'Ward'})
        break

print(f'Loaded {len(wards)} ward polygons')
print(f'Sample ward names: {wards["Ward"].head(5).tolist()}')
print()
print('Check these match the "Ward" column in the borehole dataset.')
print('If mismatches exist, run Notebook 06 first to get the crosswalk table.')

In [ ]:
# ── Ward-level zonal statistics ────────────────────────────────────────────────

def zonal(raster_path, gdf):
    stats = rasterstats.zonal_stats(
        gdf, raster_path,
        stats=['mean', 'std'], nodata=np.nan
    )
    return pd.DataFrame(stats)

print('Computing zonal statistics...')

wasi_z = zonal(wasi_path, wards)
wards['WASI_mean'] = wasi_z['mean'].round(3)
wards['WASI_std']  = wasi_z['std'].round(3)

component_labels = [
    ('C1_Distance',   'c1_distance'),
    ('C2_Rainfall',   'c2_rainfall'),
    ('C3_NDVI',       'c3_ndvi'),
    ('C4_Population', 'c4_population'),
    ('C5_Slope',      'c5_slope'),
]
for col, suffix in component_labels:
    z = zonal(OUT + f'kitui_wasi_{suffix}.tif', wards)
    wards[col] = z['mean'].round(3)

# Stress classification
def classify(s):
    if pd.isna(s):    return 'No data'
    if s >= 0.70:     return 'Very High'
    elif s >= 0.55:   return 'High'
    elif s >= 0.40:   return 'Moderate'
    elif s >= 0.25:   return 'Low'
    return 'Very Low'

wards['Stress_Class'] = wards['WASI_mean'].apply(classify)

# Functional borehole count per ward
bh_per_ward = (
    gdf_func.groupby('Ward').size()
    .rename('Functional_BH_Count').reset_index()
)
wards = wards.merge(bh_per_ward, on='Ward', how='left')
wards['Functional_BH_Count'] = wards['Functional_BH_Count'].fillna(0).astype(int)

print('Zonal statistics complete.')
print()
print(wards[['Ward', 'WASI_mean', 'Stress_Class', 'Functional_BH_Count']]
      .sort_values('WASI_mean', ascending=False)
      .head(20)
      .to_string(index=False))

In [ ]:
# ── Export ward outputs ────────────────────────────────────────────────────────

# CSV — all columns, for Notebook 03 and report tables
ward_csv_cols = [
    'Ward', 'WASI_mean', 'WASI_std', 'Stress_Class',
    'C1_Distance', 'C2_Rainfall', 'C3_NDVI', 'C4_Population', 'C5_Slope',
    'Functional_BH_Count'
]
ward_table = wards[ward_csv_cols].sort_values('WASI_mean', ascending=False)
ward_table.to_csv(OUT + 'kitui_wasi_ward_table.csv', index=False)
print(f'Ward table:  {OUT}kitui_wasi_ward_table.csv')

# GeoJSON — geometry + key columns, for Streamlit app and Notebook 03
geojson_cols = ['Ward', 'WASI_mean', 'Stress_Class', 'Functional_BH_Count', 'geometry']
wards[geojson_cols].to_file(OUT + 'kitui_wasi_ward.geojson', driver='GeoJSON')
print(f'Ward GeoJSON: {OUT}kitui_wasi_ward.geojson')

In [ ]:
# ── Diagnostic component panel ─────────────────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Water Access Stress Index — Kitui County\nComponent maps and composite (all 0–1)',
             fontsize=14, fontweight='bold')

panels = [
    (wasi,    'WASI Composite (weighted)', True),
    (c1_dist, 'C1: Distance to BH (30%)', False),
    (c2_rain, 'C2: Rainfall Deficit (25%)', False),
    (c3_ndvi, 'C3: NDVI Anomaly (15%)', False),
    (c4_pop,  'C4: Population (20%)', False),
    (c5_slope,'C5: Slope (10%)', False),
]
ext = [BOUNDS['west'], BOUNDS['east'], BOUNDS['south'], BOUNDS['north']]

for ax, (arr, title, bold) in zip(axes.flat, panels):
    im = ax.imshow(arr, cmap='RdYlGn_r', vmin=0, vmax=1,
                   extent=ext, origin='upper', aspect='equal')
    ax.set_title(title, fontsize=10, fontweight='bold' if bold else 'normal')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
                 label='0 = low stress  |  1 = high stress')

plt.tight_layout()
plt.savefig(OUT + 'kitui_wasi_diagnostic.png', dpi=150, bbox_inches='tight')
plt.show()
print('Diagnostic panel saved')

In [ ]:
# ── Ward choropleth ────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(12, 14))

wards.plot(
    column='WASI_mean', cmap='RdYlGn_r', vmin=0, vmax=1,
    linewidth=0.5, edgecolor='white', legend=True,
    legend_kwds={'label': 'WASI (0 = low stress, 1 = high stress)',
                 'orientation': 'vertical'},
    ax=ax
)

# Label the 10 highest-stress wards
for _, row in wards.nlargest(10, 'WASI_mean').iterrows():
    c = row.geometry.centroid
    ax.annotate(row['Ward'], xy=(c.x, c.y), fontsize=6.5,
                ha='center', va='center', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.15', fc='white', alpha=0.6, ec='none'))

# Functional borehole overlay
gdf_func.plot(ax=ax, color='#0B5394', markersize=4, alpha=0.55,
              label=f'GPS-verified functional BH (n={len(gdf_func)})')

ax.set_title(
    'Water Access Stress Index — Kitui County (Ward Level)\n'
    'C1×30% + C2×25% + C3×15% + C4×20% + C5×10%',
    fontsize=12, fontweight='bold'
)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(loc='lower left', fontsize=9)

plt.tight_layout()
plt.savefig(OUT + 'kitui_wasi_ward_choropleth.png', dpi=200, bbox_inches='tight')
plt.show()

print('Ward choropleth saved')
print()
print('── Notebook 02 complete ──────────────────────────────────────────────────')
print(f'Outputs in: {OUT}')
print('  kitui_wasi_500m.tif')
print('  kitui_wasi_c1_distance.tif … kitui_wasi_c5_slope.tif')
print('  kitui_wasi_ward_table.csv')
print('  kitui_wasi_ward.geojson')
print()
print('Next: Run Notebook 03 — Hotspot Analysis (Gi* clustering)')